# BEVFormerRadar — Colab Training
GPU: Runtime > Change runtime type > T4 GPU (free) or A100 (Pro)

**순서:** 1→2→3→4→5 순서대로 실행. 처음 실행 시 Cell 2(gsplat 설치)에서 ~5분 소요.

## Cell 1. Google Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── 경로 설정 (Drive 구조에 맞게 수정) ─────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/SensorFusion'       # Drive 업로드 위치
NUSCENES_ROOT = f'{DRIVE_ROOT}/data/v1.0-mini'            # nuScenes 데이터
SYNTH_DIR   = f'{DRIVE_ROOT}/NeuralSensorSim/outputs/synthetic/scene_00'
WORK_DIR    = '/content/BEVFormerRadar'                   # Colab 작업 디렉토리
CKPT_ROOT   = f'{DRIVE_ROOT}/checkpoints'                 # 체크포인트 → Drive 저장

print('Drive mount OK')
print(f'nuScenes : {NUSCENES_ROOT}  exists={os.path.isdir(NUSCENES_ROOT)}')
print(f'Synth    : {SYNTH_DIR}  exists={os.path.isdir(SYNTH_DIR)}')

## Cell 2. 코드 클론 + 의존성 설치
> gsplat은 CUDA 컴파일 필요 — 약 5분 소요.

In [ ]:
import subprocess, sys

# 코드 클론
if not os.path.isdir(WORK_DIR):
    subprocess.run(['git', 'clone',
                    'https://github.com/sjang1594/BEVFormerRadar.git',
                    WORK_DIR], check=True)
else:
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=True)

os.chdir(WORK_DIR)
print(f'Working dir: {os.getcwd()}')

# 의존성 설치
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
print('Dependencies installed')

## Cell 3. config.yaml 경로 패치
Colab 경로로 dataroot를 덮어씀 (파일은 수정하지 않음).

In [ ]:
import yaml

with open('config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataroot'] = NUSCENES_ROOT

with open('config.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print(f"dataroot → {cfg['data']['dataroot']}")

## Cell 4. GPU 확인

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU            : {torch.cuda.get_device_name(0)}')
print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 5. 학습 실행
아래 `RATIO`와 `EXP`를 바꿔서 Exp A / B / C 실행.

In [ ]:
# ── 실험 선택 ──────────────────────────────────────────────────────────────
RATIO = 0        # 0 = Exp A (real only)  |  1 = Exp B (1:1)  |  3 = Exp C (1:3)
# ─────────────────────────────────────────────────────────────────────────

cmd = [
    sys.executable, 'train_mixed.py',
    '--ratio',     str(RATIO),
    '--ckpt-root', CKPT_ROOT,
]
if RATIO > 0:
    cmd += ['--synth-dir', SYNTH_DIR]

print('Command:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## Cell 6. 평가

In [ ]:
subprocess.run([
    sys.executable, 'evaluate_all.py',
    '--ckpt-root', CKPT_ROOT,
], check=True)